# Datu kopas tīrīšana

Pirms darba Google Colab jūs jau apskatījāt sākotnējo Excel datu kopu un identificējāt tajā datu kvalitātes problēmas.

Tagad soli pa solim mēģināsim daļu no šīm problēmām novērst.

Darba princips ir šāds:

1. apskatām konkrēto problēmu;
2. izlemjam, kā to risināt;
3. dažos gadījumos paši izvēlamies tīrīšanas vai standartizācijas noteikumu;
4. piemērojam izvēlēto darbību visai datu kopai;
5. redzam, kas tieši mainījās: `Pirms → Pēc`;
6. beigās lejupielādējam iztīrīto datu kopu.

> **Svarīgi:** ne visas aizdomīgās vērtības drīkst automātiski labot. Dažos gadījumos vērtību labāk atstāt nemainītu un atzīmēt pārbaudei. Sākotnējie dati tiek saglabāti atsevišķi (`df_raw`), lai tos vienmēr varētu salīdzināt ar iztīrīto versiju.

Šī darba burtnīca nav universāls automātisks datu tīrīšanas rīks. Tā palīdz piemērot konkrētus tīrīšanas noteikumus šai datu kopai. Dažos soļos noteikumi jau ir definēti, bet citos jums pašiem būs jāpieņem lēmums, piemēram, kā standartizēt kategorijas vai kādu datuma formātu izmantot.

Tie paši noteikumi tiek piemēroti visām datu kopas rindām, arī tad, ja sākotnējā Excel failā tiek pievienotas jaunas rindas. Ja parādās vērtība, kuru izvēlētais noteikums nevar droši interpretēt, to nevajag automātiski “izlabot”. Šādu vērtību var atstāt pārbaudei.

Reālā darbā datu tīrīšanas pieeja ir atkarīga no konkrētās datu kopas, problēmas un darba nosacījumiem. Tam var izmantot Excel funkcijas un Conditional Formatting, OpenRefine, Python vai citus datu apstrādes rīkus.

Atsevišķos gadījumos datu tīrīšanā var izmantot arī mākslīgo intelektu, ja tas ir atļauts. Tomēr vispirms jāpārbauda, vai datos nav sensitīvas, konfidenciālas vai personas identificējošas informācijas. Ja šādus datus nedrīkst nosūtīt publiskiem AI rīkiem, var izmantot lokālu LLM vai droši konfigurētu API.

## 0. Datu kopas ielāde

Nospiediet ▶ un izvēlieties failu **Sintetisko-datu-kopa.xlsx**.

In [ ]:
#@title Augšupielādējiet datu kopu
from google.colab import files
import pandas as pd
import numpy as np
import re
from IPython.display import display, Markdown

uploaded = files.upload()
filename = next(iter(uploaded))

raw = pd.read_excel(filename, header=None)

headers = (
    ["STUDENTA ID"] +
    [str(x).replace("\n", " ").strip() for x in raw.iloc[1, 1:].tolist()]
)

df = raw.iloc[2:].copy()
df.columns = headers
df = df.reset_index(drop=True)

# Izveidojam stabilu ieraksta identifikatoru.
# Tas sākas ar 0 un pēc rindu dzēšanas netiek pārrēķināts.
df.insert(0, "Ieraksta ID", np.arange(len(df)))

df_raw = df.copy()

def original_values(column_name):
    """Atgriež sākotnējā faila vērtības pēc stabilā Ieraksta ID."""
    lookup = df_raw.set_index("Ieraksta ID")[column_name]
    return df["Ieraksta ID"].map(lookup)

print(f"Datu kopa veiksmīgi ielādēta: {len(df)} rindas, {len(df.columns)} kolonnas.")

### Ieraksta ID

Darba laikā izveidosim kolonnu `Ieraksta ID` (0, 1, 2, 3, ...), kas sākotnēji sakrīt ar DataFrame rindu indeksu, bet pēc rindu dzēšanas netiks pārrēķināts.

`Ieraksta ID` tiek izveidots datu ielādes brīdī un pēc rindu dzēšanas netiek pārrēķināts, tāpēc tas paliek stabils visā darba laikā.

Sākotnējā Excel failā pirmais datu ieraksts atrodas **3. rindā**, tāpēc:

**Excel rindas numurs = `Ieraksta ID + 3`**

Ja sākotnējā Excel failā pievienosiet jaunu datu rindu, arī tai tiks piešķirts `Ieraksta ID` un uz to tiks piemēroti tie paši tīrīšanas un pārbaudes noteikumi.

# 1. Pilnīgi tukšās rindas

Vispirms apskatīsim rindas, kurās **nav nevienas vērtības**.

In [ ]:
#@title Parādīt pilnīgi tukšās rindas
data_columns = [c for c in df.columns if c != "Ieraksta ID"]
empty_mask = df[data_columns].isna().all(axis=1)
empty_rows = df.loc[empty_mask].copy()

print("Pilnīgi tukšo rindu skaits:", len(empty_rows))
display(empty_rows.style.hide(axis="index").format(na_rep="NaN"))


### Ko darīt?

Izvēlieties darbību. Ja rindas dzēsīsiet, rezultātā būs redzams:

- cik rindu bija pirms tam;
- kuras rindas tika dzēstas;
- cik rindu palika pēc tam.

In [ ]:
#@title Izvēlieties darbību
Darbība = "Dzēst" #@param ["Dzēst", "Atstāt"]

before_count = len(df)

if Darbība == "Dzēst":
    data_columns = [c for c in df.columns if c != "Ieraksta ID"]
    empty_mask = df[data_columns].isna().all(axis=1)
    rows_to_delete = df.loc[empty_mask].copy()

    print("DZĒŠAMĀS RINDAS:")
    display(rows_to_delete.style.hide(axis="index").format(na_rep="NaN"))

    # Ieraksta ID netiek pārrēķināts, tāpēc pēc dzēšanas numerācijā var palikt izlaisti numuri.
    df = df.loc[~empty_mask].copy().reset_index(drop=True)

    print(f"\nRindu skaits: {before_count} → {len(df)}")
    print(f"Izdzēstas rindas: {before_count - len(df)}")
else:
    print("Nekas netika mainīts.")
    print(f"Rindu skaits paliek: {len(df)}")

# 2. Lieko atstarpju un rindu pārnesumu noņemšana

Lieka atstarpe vai rindu pārnesums var atrasties ne tikai vārdā vai studenta ID, bet arī citos īsajos teksta laukos.

Tāpēc šajā solī tīrīsim **visus īsos teksta un kategoriju laukus**, piemēram:

- `STUDENTA ID`;
- vārdu un uzvārdu;
- dzimšanas vietu;
- studiju programmu;
- `Jā / Nē` lauku;
- pēdējo apmeklēto ārvalsti.

Laukus, kuriem vēlāk ir sava īpaša pārbaude (datumi, apmierinātības skala, ienākumi), un brīvā teksta citātu šajā solī nepārveidosim.

Parastā tabulā, piemēram, `Jānis ` un `Jānis` var izskatīties pilnīgi vienādi.  
Tāpēc priekšskatījumā neredzamos simbolus padarīsim redzamus:

- `·` = atstarpe;
- `↵` = rindu pārnesums;
- `⟦ ⟧` tiek izmantotas tikai priekšskatījumā, lai būtu redzamas teksta robežas.

Vispirms apskatīsim **priekšskatījumu**. Šajā solī dati vēl netiks mainīti.

In [ ]:
#@title Priekšskatījums: kas mainīsies?

def redzams_teksts(value):
    """Padara neredzamus simbolus redzamus tikai priekšskatījumā."""
    if pd.isna(value):
        return value
    s = str(value)
    return "⟦" + (
        s.replace(" ", "·")
         .replace("\t", "⇥")
         .replace("\r", "␍")
         .replace("\n", "↵")
    ) + "⟧"

def clean_short_text_value(value):
    """Notīra tikai teksta vērtības, saglabājot citus datu tipus."""
    if not isinstance(value, str):
        return value
    return re.sub(r"[\r\n\t]+", " ", value).strip()

excluded_prefixes = (
    "Dzimšanas datums",
    "Apmierinātība ar dzīvi",
    "Vēlamie ikmēneša ienākumi",
    "Svarīgs citāts"
)

text_columns = [
    c for c in df.columns
    if c != "Ieraksta ID"
    and not c.startswith(excluded_prefixes)
]

changes = []

for col in text_columns:
    old = df[col].copy()
    new = old.map(clean_short_text_value)

    mask = old.notna() & (
        old.astype("string") != new.astype("string")
    )

    for idx in df.index[mask]:
        changes.append({
            "Ieraksta ID": df.loc[idx, "Ieraksta ID"],
            "Kolonna": col,
            "Pirms": redzams_teksts(old.loc[idx]),
            "Pēc": redzams_teksts(new.loc[idx])
        })

changes_df = pd.DataFrame(changes)

print("ŠAJĀ SOLĪ VĒL NEKAS NETIEK MAINĪTS.")
print("Tabula rāda priekšskatījumu: ko tīrīšana mainīs.")
print("· = atstarpe, ↵ = rindu pārnesums")
print("Zemāk redzama daļa no tām:")

if len(changes_df) == 0:
    print("Šāda tipa problēmas netika atrastas.")
else:
    display(changes_df.style.hide(axis="index").format(na_rep="NaN"))

### Ko darīt?

Ja izvēlaties **Notīrīt**, tiks mainītas tikai tās vērtības, kurām ir liekas atstarpes vai rindu pārnesumi.

Pēc darbības vēlreiz redzēsiet visas izmaiņas `Pirms → Pēc`.

In [ ]:
#@title Piemērot tīrīšanu?
Darbība = "Notīrīt" #@param ["Notīrīt", "Atstāt"]

if Darbība == "Notīrīt":
    all_changes = []

    for col in text_columns:
        old = df[col].copy()
        new = old.map(clean_short_text_value)

        mask = old.notna() & (
            old.astype("string") != new.astype("string")
        )

        for idx in df.index[mask]:
            all_changes.append({
                "Ieraksta ID": df.loc[idx, "Ieraksta ID"],
                "Kolonna": col,
                "Pirms": redzams_teksts(old.loc[idx]),
                "Pēc": redzams_teksts(new.loc[idx])
            })

        df[col] = new

    result = pd.DataFrame(all_changes)

    print(f"TĪRĪŠANA PIEMĒROTA. Mainīto vērtību skaits: {len(result)}")

    if len(result) > 0:
        display(result.style.hide(axis="index").format(na_rep="NaN"))

    remaining = 0
    for col in text_columns:
        old = df[col].copy()
        cleaned = old.map(clean_short_text_value)
        remaining += (
            old.notna()
            & (old.astype("string") != cleaned.astype("string"))
        ).sum()

    print(f"\nPēc tīrīšanas šajos laukos palikušas šāda tipa problēmas: {remaining}")
else:
    print("Nekas netika mainīts.")

# 3. `Jā / Nē` standartizācija

Vienai un tai pašai atbildei var būt vairāki pieraksta veidi.

Pārveidošanas noteikumus **neierakstīsim kodā iepriekš**. Notebook automātiski atradīs visas unikālās vērtības šajā kolonnā, bet jūs paši izvēlēsieties, uz kādu vērtību katru no tām pārveidot:

- `Jā`
- `Nē`
- `Atstāt / CHECK`

Ja vērtību var droši standartizēt, galvenajā kolonnā paliks tikai `Jā` vai `Nē`. Ja neesat pārliecināti un izvēlaties `Atstāt / CHECK`, galvenajā iztīrītajā kolonnā vērtība tiks atstāta tukša, bet sākotnējā vērtība tiks saglabāta kolonnā `abroad_original`.

Tas nozīmē, ka arī jauna, iepriekš neredzēta vērtība automātiski parādīsies izvēlnē, un jums būs jāizlemj, kā to apstrādāt.

In [ ]:
#@title Parādīt visas Jā / Nē vērtības
col_abroad = [c for c in df.columns if c.startswith("Vai ir studējusi ārzemēs?")][0]

display(
    df[col_abroad]
      .value_counts(dropna=False)
      .rename_axis("Vērtība")
      .to_frame("Skaits")
)

Vispirms apskatiet, kādas unikālās vērtības dators redz šajā kolonnā.

Pēc tam nākamajā solī katrai vērtībai izvēlieties atbilstošo standartizēto formu.  
Ja neesat pārliecināti, izvēlieties **`Atstāt / CHECK`**.

In [ ]:
#@title Izvēlēties vērtību pārveidojumus
import ipywidgets as widgets
from IPython.display import display, HTML

# Colab atbalsts ipywidgets interaktīvajiem elementiem
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

# Saglabājam pašreizējās vērtības.
abroad_before_mapping = df[col_abroad].astype("string").copy()

# Visas unikālās netukšās vērtības, kas reāli ir datu kopā.
unique_values = sorted(
    abroad_before_mapping.dropna().unique().tolist(),
    key=lambda x: str(x).lower()
)

mapping_widgets = {}

print("Katrai vērtībai izvēlieties, uz kādu vērtību to pārveidot:")
print()

rows = []

for value in unique_values:
    dropdown = widgets.Dropdown(
        options=[
            ("— izvēlieties —", ""),
            ("Jā", "Jā"),
            ("Nē", "Nē"),
            ("Atstāt / CHECK", "CHECK")
        ],
        value="",
        description="",
        layout=widgets.Layout(width="220px")
    )

    mapping_widgets[value] = dropdown

    value_label = widgets.HTML(
        value=f"<code>{str(value)}</code>",
        layout=widgets.Layout(width="220px")
    )

    rows.append(
        widgets.HBox(
            [value_label, dropdown],
            layout=widgets.Layout(
                align_items="center",
                margin="2px 0"
            )
        )
    )

if rows:
    display(
        widgets.VBox([
            widgets.HBox([
                widgets.HTML(
                    "<b>Sākotnējā vērtība</b>",
                    layout=widgets.Layout(width="220px")
                ),
                widgets.HTML(
                    "<b>Pārveidot uz</b>",
                    layout=widgets.Layout(width="220px")
                )
            ]),
            *rows
        ])
    )
else:
    print("Kolonnā nav aizpildītu vērtību.")

In [ ]:
#@title Piemērot izvēlētos pārveidojumus
not_selected = [
    value for value, widget in mapping_widgets.items()
    if widget.value == ""
]

if not_selected:
    raise ValueError(
        "Vispirms izvēlieties pārveidojumu visām vērtībām un pēc tam palaidiet šo šūnu vēlreiz."
    )

old = abroad_before_mapping.copy()
new = pd.Series(pd.NA, index=df.index, dtype="string")
status = pd.Series("MISSING", index=df.index, dtype="string")
selected_transformations = {}

for original_value, widget in mapping_widgets.items():
    choice = widget.value

    if choice in ("Jā", "Nē"):
        selected_transformations[original_value] = choice
    else:
        selected_transformations[original_value] = None

for original_value, target_value in selected_transformations.items():
    mask = old == original_value

    if target_value is None:
        # Neskaidra vērtība netiek ielikta tīrajā kategoriju kolonnā.
        status.loc[mask] = "CHECK"
    else:
        new.loc[mask] = target_value
        status.loc[mask] = "OK"

missing_mask = old.isna()
status.loc[missing_mask] = "MISSING"

# Saglabājam vērtību tieši tādu, kāda tā bija sākotnējā Excel failā.
if "abroad_original" not in df.columns:
    df["abroad_original"] = original_values(col_abroad)

df[col_abroad] = new
df["abroad_check"] = status

show_mask = (
    (old != new).fillna(True)
    | (status == "CHECK")
) & ~missing_mask

comparison = pd.DataFrame({
    "Ieraksta ID": df.loc[show_mask, "Ieraksta ID"],
    "Pirms": old.loc[show_mask],
    "Pēc": new.loc[show_mask],
    "Statuss": status.loc[show_mask]
})

print("IZVĒLĒTIE PĀRVEIDOŠANAS NOTEIKUMI:")

transformation_table = pd.DataFrame([
    {
        "Sākotnējā vērtība": original_value,
        "Pārveidot uz": (
            target_value
            if target_value is not None
            else "Atstāt / CHECK"
        )
    }
    for original_value, target_value in selected_transformations.items()
])

display(
    transformation_table.style
    .hide(axis="index")
    .format(na_rep="NaN")
)

print("\nMAINĪTĀS VAI PĀRBAUDĀMĀS VĒRTĪBAS:")
if len(comparison) > 0:
    display(
        comparison.style
        .hide(axis="index")
        .format(na_rep="NaN")
    )
else:
    print("Nav.")

print("\nStatusu skaits:")
display(
    df["abroad_check"]
      .value_counts(dropna=False)
      .rename_axis("Statuss")
      .to_frame("Skaits")
)

print("\nVĒRTĪBAS PĒC TĪRĪŠANAS:")
display(
    df[col_abroad]
      .value_counts(dropna=False)
      .rename_axis("Vērtība")
      .to_frame("Skaits")
)

# Kad esat izdarījuši izvēli visām vērtībām, palaidiet nākamo šūnu.

# 4. Dzimšanas datumu standartizācija

Kolonnā **Dzimšanas datums** datumi ir pierakstīti dažādos veidos, piemēram:

- `6.7.2006`
- `13.09.1993.`
- `22/02/1956`
- Excel datuma vērtības
- jau atpazīti datumi

Te pārbaudīsim divas dažādas lietas:

1. **Sintaktisko derīgumu** — vai vērtību vispār var interpretēt kā datumu?
2. **Semantisko ticamību** — vai datums šīs datu kopas kontekstā izskatās ticams?

Pirms standartizācijas jums pašiem jāizvēlas **vienots mērķa datuma formāts**.

Galvenajā `Dzimšanas datums` kolonnā pēc tīrīšanas būs tikai standartizēti `OK` datumi. Vērtības ar `CHECK` tur paliks tukšas, bet sākotnējais ieraksts tiks saglabāts kolonnā `birth_date_original`.

### Izvēlieties vienotu datuma formātu

Izvēlieties, kādā formātā datumus saglabāt pēc standartizācijas.

Piemēri:

- `dd.mm.yyyy` → `31.12.2000`
- `yyyy-mm-dd` → `2000-12-31`
- `dd/mm/yyyy` → `31/12/2000`
- `mm/dd/yyyy` → `12/31/2000`

Izvēlieties formātu, kas atbilst šīs datu kopas nosacījumiem.

In [ ]:
#@title Izvēlieties mērķa datuma formātu
Mērķa_formāts = "— izvēlieties —" #@param ["— izvēlieties —", "dd.mm.yyyy", "dd.mm.yyyy.", "yyyy-mm-dd", "dd/mm/yyyy", "mm/dd/yyyy"]

format_map = {
    "dd.mm.yyyy": "%d.%m.%Y",
    "dd.mm.yyyy.": "%d.%m.%Y.",
    "yyyy-mm-dd": "%Y-%m-%d",
    "dd/mm/yyyy": "%d/%m/%Y",
    "mm/dd/yyyy": "%m/%d/%Y"
}

if Mērķa_formāts == "— izvēlieties —":
    raise ValueError(
        "Vispirms izvēlieties mērķa datuma formātu."
    )

TARGET_DATE_FORMAT = format_map[Mērķa_formāts]

print(f"Izvēlētais mērķa formāts: {Mērķa_formāts}")
print("Piemērs:", pd.Timestamp("2003-02-05").strftime(TARGET_DATE_FORMAT))

In [ ]:
#@title Priekšskatījums: kā datumi tiks pārveidoti?
from datetime import datetime, date

col_date = [c for c in df.columns if c.startswith("Dzimšanas datums")][0]

# Šajā studentu datu kopā gads pirms 1900. gada ir pietiekami
# aizdomīgs, lai to neuzskatītu par automātiski drošu.
MIN_BIRTH_YEAR = 1900
TODAY = pd.Timestamp.today().normalize()

def parse_birth_date(value):
    if pd.isna(value):
        return pd.NaT

    if isinstance(value, (pd.Timestamp, datetime, date)):
        return pd.Timestamp(value)

    # Excel serial number
    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        try:
            return pd.Timestamp("1899-12-30") + pd.to_timedelta(float(value), unit="D")
        except Exception:
            return pd.NaT

    s = str(value).strip()
    s = re.sub(r"\s*\.\s*", ".", s)
    s = s.rstrip(".")

    for fmt in ("%d.%m.%Y", "%d/%m/%Y"):
        try:
            return pd.to_datetime(s, format=fmt, errors="raise")
        except Exception:
            pass

    return pd.NaT

def birth_date_status(original, parsed):
    if pd.isna(original):
        return "MISSING", ""

    if pd.isna(parsed):
        return "CHECK", "Vērtību nevar interpretēt kā derīgu datumu"

    if parsed.year < MIN_BIRTH_YEAR:
        return "CHECK", f"Aizdomīgs gads: {parsed.year} (< {MIN_BIRTH_YEAR})"

    if parsed > TODAY:
        return "CHECK", "Dzimšanas datums ir nākotnē"

    return "OK", ""


parsed_dates = df[col_date].apply(parse_birth_date)

statuses = []
notes = []
for original, parsed in zip(df[col_date], parsed_dates):
    status, note = birth_date_status(original, parsed)
    statuses.append(status)
    notes.append(note)

status_series = pd.Series(statuses, index=df.index, dtype="string")
cleaned_dates = parsed_dates.dt.strftime(TARGET_DATE_FORMAT)
cleaned_dates = cleaned_dates.where(status_series == "OK", pd.NA)

preview = pd.DataFrame({
    "Ieraksta ID": df["Ieraksta ID"],
    "Pirms": df[col_date],
    "Pēc": cleaned_dates,
    "Statuss": status_series,
    "Piezīme": notes
})

before_as_text = df[col_date].astype("string").str.strip()
after_as_text = preview["Pēc"].astype("string")

mask_changed = (
    (preview["Statuss"] == "CHECK")
    | (
        (preview["Statuss"] == "OK")
        & (before_as_text != after_as_text)
    )
)

display(preview.loc[mask_changed].style.hide(axis="index").format(na_rep="NaN"))

### Ko šeit nozīmē `CHECK`?

`CHECK` var nozīmēt divas atšķirīgas problēmas.

**1. Datums nav derīgs vai nav atpazīstams**

- `29.02.1987` — 1987. gads nebija garais gads;
- `18.18.1992.` — 18. mēnesis neeksistē.

**2. Datums tehniski ir derīgs, bet semantiski aizdomīgs**

Piemēram, `13.09.1704` ir korekts kalendāra datums, bet studentu datu kopā šāds dzimšanas gads ir acīmredzami aizdomīgs.

Šādas vērtības nevajag automātiski pārrakstīt uz kādu “pareizāku” datumu. Galvenajā iztīrītajā datuma kolonnā tās paliks tukšas, bet sākotnējais ieraksts būs saglabāts `birth_date_original`.

In [ ]:
#@title Piemērot datumu standartizāciju?
Darbība = "Standartizēt" #@param ["Standartizēt", "Atstāt"]

if Darbība == "Standartizēt":
    old_dates = df[col_date].copy()
    parsed_dates = df[col_date].apply(parse_birth_date)

    statuses = []
    notes = []
    for original, parsed in zip(old_dates, parsed_dates):
        status, note = birth_date_status(original, parsed)
        statuses.append(status)
        notes.append(note)

    status_series = pd.Series(statuses, index=df.index, dtype="string")

    if "birth_date_original" not in df.columns:
        df["birth_date_original"] = original_values(col_date)

    cleaned_dates = parsed_dates.dt.strftime(TARGET_DATE_FORMAT)
    cleaned_dates = cleaned_dates.where(status_series == "OK", pd.NA)

    df[col_date] = cleaned_dates
    df["birth_date_check"] = status_series
    df["birth_date_note"] = notes

    comparison = pd.DataFrame({
        "Ieraksta ID": df["Ieraksta ID"],
        "Pirms": old_dates,
        "Pēc": df[col_date],
        "Statuss": df["birth_date_check"],
        "Piezīme": df["birth_date_note"]
    })

    before_as_text = old_dates.astype("string").str.strip()
    after_as_text = df[col_date].astype("string")

    mask_show = (
        (comparison["Statuss"] == "CHECK")
        | (
            (comparison["Statuss"] == "OK")
            & (before_as_text != after_as_text)
        )
    )

    print("DATUMU STANDARTIZĀCIJA PIEMĒROTA")
    display(comparison.loc[mask_show].style.hide(axis="index").format(na_rep="NaN"))

    print("\nStatusu skaits:")
    display(
        df["birth_date_check"]
          .value_counts(dropna=False)
          .to_frame("Skaits")
    )

    print("\nCHECK vērtības galvenajā datuma kolonnā nav izmantotas; sākotnējās vērtības saglabātas birth_date_original.")
else:
    print("Dzimšanas datumi nav mainīti.")

# 5. Apmierinātība ar dzīvi un `1–5`

Kolonnā **Apmierinātība ar dzīvi** atļautas tikai vērtības:

`1, 2, 3, 4, 5`

Atradīsim ierakstus, kuri šim noteikumam neatbilst.

Pēc tīrīšanas galvenajā kolonnā būs tikai skaitliskas vērtības `1–5`. Nākamajā solī varēsiet izvēlēties, vai problemātiskās skaitliskās vērtības noapaļot, atzīmēt pārbaudei vai atstāt bez izmaiņām. Ja vērtība tiek pārveidota, tās sākotnējā forma tiks saglabāta kolonnā `satisfaction_original`.

In [ ]:
#@title Atrast problemātiskās vērtības
col_sat = [c for c in df.columns if c.startswith("Apmierinātība ar dzīvi")][0]

sat_numeric = pd.to_numeric(
    df[col_sat]
      .astype("string")
      .str.replace(",", ".", regex=False)
      .str.strip(),
    errors="coerce"
)

valid = sat_numeric.isin([1, 2, 3, 4, 5])

problematic_mask = ~valid & df[col_sat].notna()

problematic_sat = pd.DataFrame({
    "Ieraksta ID": df.loc[problematic_mask, "Ieraksta ID"],
    "Sākotnējā vērtība": df.loc[problematic_mask, col_sat],
    "Nolasītā skaitliskā vērtība": sat_numeric.loc[problematic_mask]
})

display(problematic_sat.style.hide(axis="index").format(na_rep="NaN"))

### Vai šīs vērtības drīkst automātiski noapaļot?

Piemēram:

`4,4 → 4`

Šādu vērtību var noapaļot pēc matemātiskajiem noapaļošanas principiem, taču tā vairs nebūs tieši tā pati vērtība, ko respondents sākotnēji ievadīja.

Tāpēc noapaļoto vērtību saglabāsim galvenajā analīzes kolonnā, bet sākotnējo vērtību saglabāsim atsevišķā kolonnā `satisfaction_original`.

Tas ļauj izmantot standartizētu vērtību analīzē, vienlaikus nezaudējot sākotnējo informāciju.

In [ ]:
#@title Izvēlieties darbību
Darbība = "Noapaļot" #@param ["Noapaļot", "Atzīmēt pārbaudei", "Atstāt bez izmaiņām"]

if Darbība == "Noapaļot":
    if "satisfaction_original" not in df.columns:
        df["satisfaction_original"] = original_values(col_sat)

    old_sat = df[col_sat].copy()

    # Derīgas veselās vērtības 1–5
    valid_integer = sat_numeric.isin([1, 2, 3, 4, 5])

    # Noapaļojam tikai skaitliskas vērtības intervālā 1–5.
    # 0.5 tiek noapaļots uz augšu: 2.5 → 3, 4.5 → 5.
    can_round = (
        sat_numeric.notna()
        & sat_numeric.between(1, 5)
        & ~valid_integer
    )

    rounded = np.floor(sat_numeric + 0.5)

    clean_sat = pd.Series(pd.NA, index=df.index, dtype="Int64")
    clean_sat.loc[valid_integer] = sat_numeric.loc[valid_integer].astype("Int64")
    clean_sat.loc[can_round] = rounded.loc[can_round].astype("Int64")

    df[col_sat] = clean_sat

    df["satisfaction_check"] = "CHECK"
    df.loc[df[col_sat].notna(), "satisfaction_check"] = "OK"
    df.loc[can_round, "satisfaction_check"] = "ROUNDED"
    df.loc[old_sat.isna(), "satisfaction_check"] = "MISSING"

    changed_mask = can_round | (df["satisfaction_check"] == "CHECK")

    comparison = pd.DataFrame({
        "Ieraksta ID": df.loc[changed_mask, "Ieraksta ID"],
        "Pirms": old_sat.loc[changed_mask],
        "Pēc": df.loc[changed_mask, col_sat],
        "Statuss": df.loc[changed_mask, "satisfaction_check"]
    })

    print("NOAPAĻOŠANA PIEMĒROTA.")
    print("Sākotnējās vērtības saglabātas kolonnā satisfaction_original.")
    display(
        comparison.style
        .hide(axis="index")
        .format(na_rep="NaN")
    )

elif Darbība == "Atzīmēt pārbaudei":
    if "satisfaction_original" not in df.columns:
        df["satisfaction_original"] = original_values(col_sat)

    df["satisfaction_check"] = np.where(
        df[col_sat].isna(),
        "MISSING",
        np.where(valid, "OK", "CHECK")
    )

    old_sat = df[col_sat].copy()

    clean_sat = sat_numeric.where(valid, pd.NA).astype("Int64")
    df[col_sat] = clean_sat

    flagged = pd.DataFrame({
        "Ieraksta ID": df.loc[df["satisfaction_check"] == "CHECK", "Ieraksta ID"],
        "Pirms": old_sat.loc[df["satisfaction_check"] == "CHECK"],
        "Pēc": df.loc[df["satisfaction_check"] == "CHECK", col_sat],
        "Statuss": df.loc[df["satisfaction_check"] == "CHECK", "satisfaction_check"]
    })

    print("PROBLEMĀTISKĀS VĒRTĪBAS NETIKA AUTOMĀTISKI LABOTAS.")
    display(
        flagged.style
        .hide(axis="index")
        .format(na_rep="NaN")
    )

else:
    print("Nekas netika mainīts.")

# 6. Ienākumu lauks

Dažās rindās skaitlis ir sajaukts ar papildu tekstu:

- `5000 eiro`
- `1876(neto)`
- `6572(bruto)`
- `~3000`

Tā kā sākotnējā kolonna jau ir paredzēta **skaitlim eiro**, pēc tīrīšanas tieši šajā galvenajā kolonnā saglabāsim skaitlisko vērtību.

Sākotnējais ieraksts tiks saglabāts `income_original`, papildu informācija, piemēram, `neto`, `bruto` vai `aptuvens`, būs `income_note`, bet pārbaudes statuss būs `income_check`.

Sarežģītu vērtību, piemēram, `5000–6000`, nedrīkst automātiski pārvērst par `5000`. Šādos gadījumos galvenajā ienākumu kolonnā paliks tukša vērtība un `income_check = CHECK`.

In [ ]:
#@title Parādīt nestandarta ienākumu vērtības
col_income = [c for c in df.columns if c.startswith("Vēlamie ikmēneša ienākumi")][0]

def parse_income(value):
    if pd.isna(value):
        return np.nan, "", "MISSING"

    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        return float(value), "", "OK"

    s = str(value).strip()
    if s == "":
        return np.nan, "", "MISSING"

    # Meklējam visus skaitļus. Ja ir vairāk par vienu,
    # piemēram, 5000–6000, automātiski neizvēlamies vienu no tiem.
    number_matches = re.findall(r"\d+(?:[.,]\d+)?", s)

    if len(number_matches) != 1:
        return np.nan, "sarežģīts vai neatpazīts formāts", "CHECK"

    number_text = number_matches[0]
    amount = pd.to_numeric(number_text.replace(",", "."), errors="coerce")

    lower = s.lower()

    has_neto = "neto" in lower
    has_bruto = "bruto" in lower

    if has_neto and has_bruto:
        return np.nan, "vienlaikus norādīts neto un bruto", "CHECK"

    notes = []
    if has_neto:
        notes.append("neto")
    if has_bruto:
        notes.append("bruto")

    if (
        "~" in lower
        or re.search(r"\baptuveni\b", lower)
        or re.search(r"\bap\b", lower)
    ):
        notes.append("aptuvens")

    # Pārbaudām, vai bez skaitļa nepaliek nesaprotams teksts.
    remainder = lower
    remainder = remainder.replace(number_text.lower(), "", 1)

    for token in ["eiro", "eur", "€", "neto", "bruto", "aptuveni", "ap", "~"]:
        remainder = remainder.replace(token, "")

    remainder = re.sub(r"[\s()]+", "", remainder)

    # Defise bez otra skaitļa arī nav pietiekami skaidra.
    if remainder != "":
        return np.nan, "sarežģīts vai neatpazīts formāts", "CHECK"

    return float(amount), ", ".join(notes), "OK"

parsed_income = df[col_income].apply(parse_income)

income_preview = pd.DataFrame({
    "Ieraksta ID": df["Ieraksta ID"],
    "Pirms": df[col_income],
    "Pēc": parsed_income.map(lambda x: x[0]),
    "Pēc: income_note": parsed_income.map(lambda x: x[1]),
    "Statuss": parsed_income.map(lambda x: x[2])
})

# Rādām tās vērtības, kas nav vienkārši tīrs skaitlis,
# kā arī visas CHECK vērtības.
original_as_text = df[col_income].astype("string")
simple_number = original_as_text.str.fullmatch(r"\s*\d+(?:[.,]\d+)?\s*", na=False)

show_mask = (~simple_number & df[col_income].notna()) | (income_preview["Statuss"] == "CHECK")

display(income_preview.loc[show_mask].style.hide(axis="index").format(na_rep="NaN"))

In [ ]:
#@title Izveidot strukturētus ienākumu laukus
old_income = df[col_income].copy()
parsed_income = df[col_income].apply(parse_income)

if "income_original" not in df.columns:
    df["income_original"] = original_values(col_income)

df["income_note"] = parsed_income.map(lambda x: x[1])
df["income_check"] = parsed_income.map(lambda x: x[2])

parsed_amount = parsed_income.map(lambda x: x[0])
df[col_income] = parsed_amount.where(df["income_check"] == "OK", np.nan)

comparison_income = pd.DataFrame({
    "Ieraksta ID": df["Ieraksta ID"],
    "Pirms": old_income,
    "Pēc": df[col_income],
    "Piezīme": df["income_note"],
    "Statuss": df["income_check"]
})

original_as_text = old_income.astype("string")
simple_number = original_as_text.str.fullmatch(r"\s*\d+(?:[.,]\d+)?\s*", na=False)
show_mask = (~simple_number & old_income.notna()) | (df["income_check"] == "CHECK")

display(comparison_income.loc[show_mask].style.hide(axis="index").format(na_rep="NaN"))

print("\nStatusu skaits:")
display(
    df["income_check"]
      .value_counts(dropna=False)
      .to_frame("Skaits")
)

check_count = (df["income_check"] == "CHECK").sum()
print(f"\nAutomātiski neinterpretētas vērtības: {check_count}")

# 7. Kopsavilkums: kas tika mainīts?

Gala datu kopā saglabāsim vienotu struktūru:

1. `Ieraksta ID`;
2. visas sākotnējās datu kolonnas **sākotnējā secībā**, bet ar iztīrītajām vērtībām;
3. beigās `_original`, `_check` un `_note` kolonnas, kas ļauj pārbaudīt veiktās izmaiņas.

In [ ]:
#@title Parādīt kopsavilkumu

core_columns = [c for c in df_raw.columns if c in df.columns]
audit_order = ['abroad_original', 'abroad_check', 'birth_date_original', 'birth_date_check', 'birth_date_note', 'satisfaction_original', 'satisfaction_check', 'income_original', 'income_note', 'income_check']
audit_order = [c for c in audit_order if c in df.columns]

other_columns = [
    c for c in df.columns
    if c not in core_columns and c not in audit_order
]

df = df[core_columns + audit_order + other_columns]

print("Sākotnējais izmērs:", df_raw.shape)
print("Pašreizējais izmērs:", df.shape)

print("\nPievienotās pārbaudes kolonnas:")
new_cols = [c for c in df.columns if c not in df_raw.columns]
print(new_cols if new_cols else "Nav")

print("\nPašreizējās datu kopas pirmās rindas:")
display(df.head(10).style.hide(axis="index").format(na_rep="NaN"))

# 8. Lejupielādēt iztīrīto datu kopu

Nospiediet ▶.

Tiks izveidots fails:

**`Sintetisko-datu-kopa_CLEANED.xlsx`**

un pārlūkprogramma piedāvās to lejupielādēt.

In [ ]:
#@title Lejupielādēt CLEANED datu kopu
from google.colab import files

core_columns = [c for c in df_raw.columns if c in df.columns]
audit_order = ['abroad_original', 'abroad_check', 'birth_date_original', 'birth_date_check', 'birth_date_note', 'satisfaction_original', 'satisfaction_check', 'income_original', 'income_note', 'income_check']
audit_order = [c for c in audit_order if c in df.columns]

other_columns = [
    c for c in df.columns
    if c not in core_columns and c not in audit_order
]

df = df[core_columns + audit_order + other_columns]

output_filename = "Sintetisko-datu-kopa_CLEANED.xlsx"

df.to_excel(output_filename, index=False)

print(f"Izveidots fails: {output_filename}")
print(f"Rindas: {len(df)}")
print(f"Kolonnas: {len(df.columns)}")

files.download(output_filename)

Piezīme: iztīrītais fails tiek izveidots no jauna, tāpēc sākotnējais Excel vizuālais formatējums netiek saglabāts. Tas neietekmē iztīrīto datu saturu.

# Noslēguma jautājumi

1. Kāpēc ir nepieciešams tīrīt datus?

2. Ko vēl varētu iztīrīt šajā datu kopā, ko mēs šajā uzdevumā neiztīrījām?

3. Kuras datu kvalitātes problēmas varēja labot automātiski, un kurām bija nepieciešama cilvēka pārbaude?

4. Kāpēc dažos gadījumos ir svarīgi saglabāt sākotnējo vērtību atsevišķā kolonnā?

5. Vai vienmēr ir nepieciešams tīrīt datus pirms analīzes?